In [ ]:
# ============================================================================
# IV CRUSH SCREENER - FINAL 6-DAY CYCLE (TUE-MON)
# ============================================================================

!pip install yfinance pandas numpy tqdm -q

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import warnings
import time

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    # Strategy settings
    ENTRY_TRADING_DAYS_BEFORE = 3   # Enter 3 days before earnings
    
    # Run settings
    INCLUDE_TODAY_IF_BEFORE_OPEN = True
    MARKET_OPEN_TIME = 9
    
    # Filters
    MIN_PRICE = 5
    MAX_PRICE = 2000
    
    MIN_IV_PERCENTILE = 30
    APPLY_IV_FILTER = False
    
    TOP_N = 10


# ============================================================================
# WATCHLIST
# ============================================================================

raw_tickers = """
A AA AAL AAP AAPL ABBV ABNB ABT ACN ADBE ADI ADM ADP ADSK AEP AES AFL AFRM AGNC AI AIG AKAM ALB ALK ALL AMAT AMD AMGN AMRN AMZN APA APH APTV AVGO AXP BA BABA BAC BAX BBY BEN BIDU BIIB BK BKR BMY BP BSX BX BYND C CAH CAT CB CCI CCJ CCL CF CFG CHWY CI CL CLF CLX CMCSA CME CNC CNP COF COIN COP COST CPB CPRI CRM CRON CRWD CSCO CSX CTVA CVNA CVS CVX CZR D DAL DD DE DELL DHI DHR DIS DKNG DLR DOCU DOW DRI DVN DXC EA EBAY ED EIX EL EMR ENPH EOG EQR EQT ETSY EVRG EW EXC EXPE F FANG FAST FCX FDX FE FITB FOXA FSLR FTI FTV GD GE GILD GLW GM GME GOOG GOOGL GPRO GPS GS HAL HBAN HCA HD HIG HLT HOG HOLX HON HPE HPQ IBM ICE INTC IP IRM IVZ JCI JD JNJ JPM K KHC KMI KO KR KSS LEN LI LKQ LLY LNC LOW LUMN LUV LVS LYB LYFT MA MAR MARA MAS MCD MCHP MDLZ MDT MET META MGM MMM MO MOS MPC MRK MRNA MRVL MS MSFT MSTR MTB MTCH MU NCLH NEE NEM NET NFLX NKE NOW NRG NTAP NTES NVAX NVDA NWL NWSA O OKE OMC ON ORCL OXY PBR PDD PENN PEP PFE PFG PG PGR PINS PLTR PNC PPL PRU PSX PTON PYPL QCOM RBLX RCL RF RIG RIOT RIVN ROKU ROST RTX RUN SBUX SCHW SEDG SHOP SIRI SLB SMCI SNAP SNOW SO SOFI SPG STX SWKS SYF SYY T TAP TCOM TDOC TFC TGT TJX TMO TMUS TPR TRIP TRV TSLA TSM TSN TT TTD TTWO TXN U UA UAA UAL UBER ULTA UNH UNP UPS UPST URBN USB V VALE VFC VTR VZ WAB WBD WDC WFC WM WMB WMT WU WY WYNN XEL YELP ZION ZM
"""

WATCHLIST = [ticker.strip() for ticker in raw_tickers.split() if ticker.strip()]

ETF_EXCLUSIONS = {
    'DIA', 'SPY', 'QQQ', 'IWM', 'EEM', 'EFA', 'GLD', 'SLV', 'GDX', 'HYG', 
    'IBB', 'IEF', 'IYR', 'KRE', 'LQD', 'OIH', 'SMH', 'TLT', 'UNG', 'USO', 
    'VGK', 'VXX', 'XBI', 'XHB', 'XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 
    'XLP', 'XLRE', 'XLU', 'XLV', 'XLY', 'XOP', 'XRT', 'SOXL', 'SOXS', 
    'SPXL', 'SPXS', 'SQQQ', 'TQQQ', 'UVXY', 'JETS', 'BITO', 'IBIT', 'KWEB', 
    'YINN', 'FXE', 'FXI', 'TBT', 'SCHD'
}

WATCHLIST = [ticker for ticker in WATCHLIST if ticker not in ETF_EXCLUSIONS]
print(f"📊 Loaded {len(WATCHLIST)} individual stocks\n")


# ============================================================================
# TRADING CALENDAR
# ============================================================================

class TradingCalendar:
    @staticmethod
    def is_trading_day(date):
        if isinstance(date, datetime):
            date = date.date()
        return date.weekday() < 5
    
    @staticmethod
    def get_next_trading_day(date):
        if isinstance(date, datetime):
            date = date.date()
        while not TradingCalendar.is_trading_day(date):
            date += timedelta(days=1)
        return date
    
    @staticmethod
    def add_trading_days(date, days):
        if isinstance(date, datetime):
            date = date.date()
        if days >= 0:
            while days > 0:
                date += timedelta(days=1)
                if TradingCalendar.is_trading_day(date):
                    days -= 1
        else:
            while days < 0:
                date -= timedelta(days=1)
                if TradingCalendar.is_trading_day(date):
                    days += 1
        return date
    
    @staticmethod
    def calculate_entry_date(earnings_date, trading_days_before):
        if isinstance(earnings_date, datetime):
            earnings_date = earnings_date.date()
        
        entry_date = earnings_date
        days_counted = 0
        
        while days_counted < trading_days_before:
            entry_date -= timedelta(days=1)
            if TradingCalendar.is_trading_day(entry_date):
                days_counted += 1
        
        return entry_date


# ============================================================================
# ENTRY WINDOW (FIXED FOR 6 DAY CYCLE: TUE-MON)
# ============================================================================

class EntryWindow:
    def __init__(self, run_datetime=None):
        self.run_datetime = run_datetime or datetime.now()
        self.run_date = self.run_datetime.date()
        self.run_hour = self.run_datetime.hour
        self.calendar = TradingCalendar()
        
        self.is_before_open = self.run_hour < Config.MARKET_OPEN_TIME
        
        # 1. Determine START of Entry Window
        if self.run_date.weekday() == 1 and self.is_before_open and Config.INCLUDE_TODAY_IF_BEFORE_OPEN:
            # If it's Tuesday morning, start today
            self.first_entry_date = self.run_date
        else:
            # Otherwise start next valid trading day
            self.first_entry_date = self.calendar.get_next_trading_day(self.run_date)
        
        # 2. Determine END of Entry Window (STRICT 6-DAY LIMIT)
        # We explicitly set this to exactly 6 calendar days from the start.
        # Example: Tuesday + 6 days = Monday.
        # This prevents overlapping with next Tuesday's scan.
        self.last_entry_date = self.first_entry_date + timedelta(days=6)
    
    def print_info(self):
        print(f"{'='*75}")
        print(f"  📅 SCAN CONFIGURATION (6-Day Cycle: Tue -> Mon)")
        print(f"{'='*75}")
        print(f"""
  Run Date/Time:        {self.run_datetime.strftime('%A, %B %d, %Y at %I:%M %p')}
  Before Market Open:   {'✅ Yes' if self.is_before_open else '❌ No'}
  
  Entry Window:         {self.first_entry_date.strftime('%a %m/%d')} - {self.last_entry_date.strftime('%a %m/%d')} (Strict 6 Days)
  Bot Entry Trigger:    {Config.ENTRY_TRADING_DAYS_BEFORE} trading days before earnings
        """)


# ============================================================================
# DATA FETCHERS
# ============================================================================

def get_earnings_date(ticker):
    """Get earnings date using multiple methods"""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        earnings_dt = None
        earnings_time = 'AMC'
        method_used = None
        
        # Method 1: earningsTimestamp
        if info.get('earningsTimestamp'):
            earnings_dt = datetime.fromtimestamp(info['earningsTimestamp'])
            earnings_time = 'BMO' if earnings_dt.hour < 12 else 'AMC'
            method_used = 'earningsTimestamp'
        
        # Method 2: calendar
        if earnings_dt is None:
            try:
                calendar = stock.calendar
                if calendar and 'Earnings Date' in calendar:
                    earnings_dates = calendar['Earnings Date']
                    if isinstance(earnings_dates, list) and len(earnings_dates) > 0:
                        ed = earnings_dates[0]
                        if hasattr(ed, 'year'):
                            earnings_dt = datetime.combine(ed, datetime.min.time()) if not isinstance(ed, datetime) else ed
                            method_used = 'calendar'
            except:
                pass
        
        # Method 3: earnings_dates DataFrame
        if earnings_dt is None:
            try:
                df = stock.earnings_dates
                if df is not None and len(df) > 0:
                    dt = df.index[0]
                    
                    if hasattr(dt, 'tz_localize'):
                        dt = dt.tz_localize(None)
                    elif hasattr(dt, 'tzinfo') and dt.tzinfo is not None:
                        dt = dt.replace(tzinfo=None)
                    
                    if hasattr(dt, 'to_pydatetime'):
                        dt = dt.to_pydatetime()
                    
                    if isinstance(dt, datetime) and dt.date() >= datetime.now().date():
                        earnings_dt = dt
                        method_used = 'earnings_dates'
            except:
                pass
        
        if earnings_dt:
            return {
                'ticker': ticker,
                'earnings_datetime': earnings_dt,
                'earnings_date': earnings_dt.date() if isinstance(earnings_dt, datetime) else earnings_dt,
                'earnings_time': earnings_time,
                'method': method_used
            }
        
        return None
        
    except:
        return None


def check_entry_window(earnings_info, window):
    """Check if earnings date falls within actionable entry window"""
    
    earnings_date = earnings_info['earnings_date']
    
    entry_date = TradingCalendar.calculate_entry_date(
        earnings_date,
        Config.ENTRY_TRADING_DAYS_BEFORE
    )
    
    if earnings_info['earnings_time'] == 'BMO':
        exit_date = earnings_date
    else:
        exit_date = TradingCalendar.add_trading_days(earnings_date, 1)
    
    # STRICT CHECK: Entry date must be within start and end date inclusive
    in_window = window.first_entry_date <= entry_date <= window.last_entry_date
    
    return {
        **earnings_info,
        'entry_date': entry_date,
        'exit_date': exit_date,
        'in_window': in_window,
        'days_until_entry': (entry_date - datetime.now().date()).days,
        'days_until_earnings': (earnings_date - datetime.now().date()).days
    }


def get_expected_move(ticker, earnings_date):
    """Get expected move from options straddle"""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        price = info.get('regularMarketPrice') or info.get('currentPrice')
        if not price:
            hist = stock.history(period='5d')
            price = hist['Close'].iloc[-1] if len(hist) > 0 else None
        if not price:
            return None
        
        expirations = stock.options
        if not expirations:
            return None
        
        target_exp = None
        for exp in expirations:
            exp_dt = datetime.strptime(exp, '%Y-%m-%d')
            if exp_dt.date() >= earnings_date:
                target_exp = exp
                break
        
        if not target_exp:
            target_exp = expirations[0]
        
        chain = stock.option_chain(target_exp)
        calls, puts = chain.calls, chain.puts
        
        if len(calls) == 0 or len(puts) == 0:
            return None
        
        def get_mid(row):
            if row['bid'] > 0 and row['ask'] > 0:
                return (row['bid'] + row['ask']) / 2
            return row['lastPrice']
        
        calls['dist'] = abs(calls['strike'] - price)
        atm_strike = calls.loc[calls['dist'].idxmin(), 'strike']
        
        atm_call = calls[calls['strike'] == atm_strike].iloc[0]
        atm_put = puts[puts['strike'] == atm_strike].iloc[0]
        
        call_mid = get_mid(atm_call)
        put_mid = get_mid(atm_put)
        straddle = call_mid + put_mid
        expected_move_pct = (straddle / price) * 100
        
        call_iv = (atm_call.get('impliedVolatility') or 0) * 100
        put_iv = (atm_put.get('impliedVolatility') or 0) * 100
        avg_iv = (call_iv + put_iv) / 2 if (call_iv > 0 and put_iv > 0) else None
        
        wing_distance = straddle
        
        long_call_candidates = calls[calls['strike'] >= atm_strike + wing_distance * 0.8]
        long_put_candidates = puts[puts['strike'] <= atm_strike - wing_distance * 0.8]
        
        if len(long_call_candidates) == 0:
            long_call_candidates = calls[calls['strike'] > atm_strike].head(3)
        if len(long_put_candidates) == 0:
            long_put_candidates = puts[puts['strike'] < atm_strike].tail(3)
        
        if len(long_call_candidates) > 0 and len(long_put_candidates) > 0:
            long_call = long_call_candidates.iloc[0]
            long_put = long_put_candidates.iloc[-1]
            
            long_call_price = get_mid(long_call)
            long_put_price = get_mid(long_put)
            
            credit = (call_mid + put_mid) - (long_call_price + long_put_price)
            max_loss = max(long_call['strike'] - atm_strike, atm_strike - long_put['strike']) - credit
        else:
            long_call = None
            long_put = None
            credit = None
            max_loss = None
        
        return {
            'price': price,
            'atm_strike': atm_strike,
            'straddle_price': straddle,
            'expected_move_pct': expected_move_pct,
            'avg_iv': avg_iv,
            'expiry': target_exp,
            'long_call_strike': long_call['strike'] if long_call is not None else None,
            'long_put_strike': long_put['strike'] if long_put is not None else None,
            'credit': credit,
            'max_loss': max_loss
        }
    except:
        return None


def get_historical_moves(ticker):
    """Get historical earnings moves"""
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period='2y')
        
        if len(hist) < 100:
            return None
        
        hist['return'] = hist['Close'].pct_change().abs() * 100
        hist['quarter'] = hist.index.to_period('Q')
        quarterly_max = hist.groupby('quarter')['return'].max()
        moves = quarterly_max.dropna().tail(8).tolist()
        
        if len(moves) >= 4:
            return {
                'avg': np.mean(moves),
                'median': np.median(moves),
                'max': np.max(moves),
                'moves': moves
            }
        return None
    except:
        return None


def get_iv_percentile(ticker):
    """Calculate IV percentile"""
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period='1y')
        
        if len(hist) < 60:
            return None
        
        hist['ret'] = np.log(hist['Close'] / hist['Close'].shift(1))
        hist['hv20'] = hist['ret'].rolling(20).std() * np.sqrt(252) * 100
        
        current_hv = hist['hv20'].iloc[-1]
        hv_series = hist['hv20'].dropna()
        
        if len(hv_series) < 20:
            return None
        
        return (hv_series < current_hv).sum() / len(hv_series) * 100
    except:
        return None


# ============================================================================
# MAIN SCREENER
# ============================================================================

def run_screener():
    """Run screener with step-by-step filter reporting"""
    
    print(f"""
    ╔═══════════════════════════════════════════════════════════════╗
    ║                                                               ║
    ║       IV CRUSH SCREENER - 6 DAY FILTER (TUE-MON)             ║
    ║                                                               ║
    ╚═══════════════════════════════════════════════════════════════╝
    """)
    
    window = EntryWindow()
    window.print_info()
    
    # =====================================================================
    # STEP 1: Get ALL earnings dates
    # =====================================================================
    
    print(f"\n{'='*75}")
    print(f"  STEP 1: FETCHING EARNINGS DATES FOR ALL {len(WATCHLIST)} STOCKS")
    print(f"{'='*75}\n")
    
    earnings_data = []
    no_earnings = []
    
    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(get_earnings_date, t): t for t in WATCHLIST}
        
        for future in tqdm(as_completed(futures), total=len(WATCHLIST), desc="Fetching earnings"):
            ticker = futures[future]
            result = future.result()
            if result:
                earnings_data.append(result)
            else:
                no_earnings.append(ticker)
            time.sleep(0.02)
    
    print(f"\n  ✅ Found earnings dates: {len(earnings_data)} stocks")
    print(f"  ❌ No earnings date:     {len(no_earnings)} stocks")
    
    if len(no_earnings) > 0 and len(no_earnings) <= 20:
        print(f"     ({', '.join(no_earnings)})")
    
    if len(earnings_data) == 0:
        print("\n❌ No stocks with earnings dates found!")
        return None
    
    # Sort by earnings date
    earnings_data.sort(key=lambda x: x['earnings_date'])
    
    # =====================================================================
    # STEP 2: Filter by Entry Window
    # =====================================================================
    
    print(f"\n\n{'='*75}")
    print(f"  STEP 2: FILTER BY ENTRY WINDOW")
    print(f"          Entry must be between {window.first_entry_date.strftime('%a %m/%d')} and {window.last_entry_date.strftime('%a %m/%d')}")
    print(f"{'='*75}\n")
    
    in_window = []
    outside_window = []
    
    for e in earnings_data:
        result = check_entry_window(e, window)
        if result['in_window']:
            in_window.append(result)
        else:
            outside_window.append(result)
    
    print(f"  ✅ In window:      {len(in_window)} stocks")
    print(f"  ❌ Outside window: {len(outside_window)} stocks")
    
    if len(in_window) > 0:
        print(f"\n  📅 STOCKS IN ENTRY WINDOW:")
        print(f"  {'─'*70}")
        for s in in_window:
            print(f"     {s['ticker']:<6} Entry: {s['entry_date'].strftime('%a %m/%d')} → Earnings: {s['earnings_date'].strftime('%a %m/%d')} ({s['earnings_time']})")
    
    if len(in_window) == 0:
        print("\n❌ No stocks in entry window!")
        return None
    
    # =====================================================================
    # STEP 3: Get Expected Move and Options Data
    # =====================================================================
    
    print(f"\n\n{'='*75}")
    print(f"  STEP 3: GETTING EXPECTED MOVE DATA")
    print(f"{'='*75}\n")
    
    with_options = []
    no_options = []
    
    for s in tqdm(in_window, desc="Getting options data"):
        options_data = get_expected_move(s['ticker'], s['earnings_date'])
        if options_data:
            with_options.append({**s, **options_data})
        else:
            no_options.append(s['ticker'])
        time.sleep(0.05)
    
    print(f"\n  ✅ Got options data: {len(with_options)} stocks")
    print(f"  ❌ No options data:  {len(no_options)} stocks")
    
    if len(no_options) > 0:
        print(f"     ({', '.join(no_options)})")
    
    if len(with_options) == 0:
        print("\n❌ No stocks with options data!")
        return None
    
    print(f"\n  📊 EXPECTED MOVES:")
    print(f"  {'─'*70}")
    for s in with_options:
        print(f"     {s['ticker']:<6} Price: ${s['price']:<8.2f} Expected Move: ±{s['expected_move_pct']:.1f}%")
    
    # =====================================================================
    # STEP 4: Get Historical Moves
    # =====================================================================
    
    print(f"\n\n{'='*75}")
    print(f"  STEP 4: GETTING HISTORICAL MOVES")
    print(f"{'='*75}\n")
    
    with_history = []
    no_history = []
    
    for s in tqdm(with_options, desc="Getting historical data"):
        hist_data = get_historical_moves(s['ticker'])
        if hist_data:
            move_ratio = hist_data['avg'] / s['expected_move_pct']
            beat_rate = sum(1 for m in hist_data['moves'] if m < s['expected_move_pct']) / len(hist_data['moves']) * 100
            with_history.append({
                **s,
                'hist_avg_move': hist_data['avg'],
                'hist_max_move': hist_data['max'],
                'move_ratio': move_ratio,
                'beat_rate': beat_rate
            })
        else:
            with_history.append({
                **s,
                'hist_avg_move': None,
                'hist_max_move': None,
                'move_ratio': None,
                'beat_rate': None
            })
            no_history.append(s['ticker'])
        time.sleep(0.02)
    
    print(f"\n  ✅ Got historical data: {len(with_history) - len(no_history)} stocks")
    print(f"  ⚠️  No historical data: {len(no_history)} stocks (still included)")
    
    print(f"\n  📊 EXPECTED vs HISTORICAL MOVES:")
    print(f"  {'─'*70}")
    print(f"  {'Ticker':<8} {'Expected':>10} {'Historical':>12} {'Ratio':>8} {'Beat Rate':>10}")
    print(f"  {'─'*70}")
    for s in with_history:
        hist_str = f"±{s['hist_avg_move']:.1f}%" if s['hist_avg_move'] else "N/A"
        ratio_str = f"{s['move_ratio']:.2f}x" if s['move_ratio'] else "N/A"
        beat_str = f"{s['beat_rate']:.0f}%" if s['beat_rate'] else "N/A"
        print(f"  {s['ticker']:<8} ±{s['expected_move_pct']:>8.1f}% {hist_str:>12} {ratio_str:>8} {beat_str:>10}")
    
    # =====================================================================
    # STEP 5: Get IV Percentile
    # =====================================================================
    
    print(f"\n\n{'='*75}")
    print(f"  STEP 5: GETTING IV PERCENTILE")
    print(f"{'='*75}\n")
    
    final_data = []
    
    for s in tqdm(with_history, desc="Getting IV percentile"):
        iv_pct = get_iv_percentile(s['ticker'])
        final_data.append({
            **s,
            'iv_percentile': iv_pct
        })
        time.sleep(0.02)
    
    print(f"\n  📊 IV PERCENTILES:")
    print(f"  {'─'*70}")
    for s in final_data:
        iv_str = f"{s['iv_percentile']:.0f}%" if s['iv_percentile'] else "N/A"
        badge = "🔥 HIGH" if s['iv_percentile'] and s['iv_percentile'] > 70 else ""
        print(f"     {s['ticker']:<6} IV Percentile: {iv_str:<10} {badge}")
    
    # =====================================================================
    # STEP 6: Calculate Final Score and Rank
    # =====================================================================
    
    print(f"\n\n{'='*75}")
    print(f"  STEP 6: FINAL SCORING AND RANKING")
    print(f"{'='*75}\n")
    
    for s in final_data:
        score = 0
        
        if s['iv_percentile']:
            score += (s['iv_percentile'] / 100) * 25
        else:
            score += 10
        
        if s['move_ratio']:
            ratio_score = max(0, min(1, 1.4 - s['move_ratio']))
            score += ratio_score * 35
        else:
            score += 15
        
        if s['beat_rate']:
            score += (s['beat_rate'] / 100) * 25
        else:
            score += 12
        
        score += 15
        
        s['score'] = round(score, 1)
    
    final_data.sort(key=lambda x: x['score'], reverse=True)
    
    return final_data[:Config.TOP_N]


# ============================================================================
# ENHANCED OUTPUT WITH TRADING RULES
# ============================================================================

def print_results_with_rules(top_stocks):
    """Print results with complete trading rules"""
    
    if not top_stocks:
        print("\n❌ No stocks found this week.")
        return None
    
    print(f"\n{'='*75}")
    print(f"  🏆 TOP {len(top_stocks)} STOCKS FOR THIS WEEK (TUE - MON)")
    print(f"{'='*75}\n")
    
    print("  ✂️  COPY & PASTE INTO YOUR BOT:")
    print("  " + "─"*70)
    tickers = [s['ticker'] for s in top_stocks]
    print(f"\n  {', '.join(tickers)}")
    print(f"\n  {tickers}\n")
    
    print("  📅 THIS WEEK'S SCHEDULE:")
    print("  " + "─"*70)
    
    by_entry = {}
    for s in top_stocks:
        entry_str = s['entry_date'].strftime('%Y-%m-%d')
        if entry_str not in by_entry:
            by_entry[entry_str] = []
        by_entry[entry_str].append(s)
    
    for entry_str in sorted(by_entry.keys()):
        stocks = by_entry[entry_str]
        entry_dt = datetime.strptime(entry_str, '%Y-%m-%d').date()
        days_away = (entry_dt - datetime.now().date()).days
        
        badge = "🔴 TODAY" if days_away == 0 else "🟡 TOMORROW" if days_away == 1 else f"🟢 in {days_away}d"
        
        print(f"\n  {entry_dt.strftime('%A %b %d')} ({badge})")
        for s in stocks:
            print(f"    → {s['ticker']:<6}  Earnings: {s['earnings_date'].strftime('%a %m/%d')} {s['earnings_time']}  Exit: {s['exit_date'].strftime('%a %m/%d')} 9:40 AM")
    
    print(f"\n\n  📊 DETAILED BREAKDOWN:")
    print("  " + "─"*70)
    
    for i, s in enumerate(top_stocks, 1):
        print(f"\n  #{i} {s['ticker']} (Score: {s['score']})")
        print(f"      Price: ${s['price']:.2f}")
        print(f"      Entry: {s['entry_date'].strftime('%a %m/%d')} | Earnings: {s['earnings_date'].strftime('%a %m/%d')} {s['earnings_time']}")
        print(f"      Expected Move: ±{s['expected_move_pct']:.1f}%", end="")
        if s.get('hist_avg_move'):
            print(f" | Historical: ±{s['hist_avg_move']:.1f}% | Ratio: {s['move_ratio']:.2f}x")
        else:
            print(" | Historical: N/A")
        if s.get('beat_rate'):
            print(f"      Beat Rate: {s['beat_rate']:.0f}%", end="")
        if s.get('iv_percentile'):
            print(f" | IV Percentile: {s['iv_percentile']:.0f}%")
        else:
            print()
        
        if s.get('credit') and s.get('max_loss'):
            print(f"      Iron Butterfly: ATM ${s['atm_strike']:.0f} | Credit: ${s['credit']:.2f} | Max Loss: ${s['max_loss']:.2f}")
    
    df = pd.DataFrame(top_stocks)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    csv_file = f'iv_crush_{timestamp}.csv'
    df.to_csv(csv_file, index=False)
    print(f"\n\n  ✅ Saved to: {csv_file}")
    
    # =========================================================================
    # COMPLETE TRADING RULES
    # =========================================================================
    
    print(f"\n\n{'='*75}")
    print(f"  📋 COMPLETE TRADING RULES & BOT CONFIGURATION")
    print(f"{'='*75}\n")
    
    print("""
  ╔═════════════════════════════════════════════════════════════════════╗
  ║  ENTRY RULES                                                        ║
  ╚═════════════════════════════════════════════════════════════════════╝

  Timing:   Enter EXACTLY 3 trading days before earnings
  Bot Setting: "Enter when earnings is exactly 3 days away"

  Iron Butterfly Structure:
    • SELL 1 ATM Call (Delta ~0.50)
    • SELL 1 ATM Put (Delta ~0.50)
    • BUY 1 OTM Call (Delta ~0.16, placed at expected move distance)
    • BUY 1 OTM Put (Delta ~0.16, placed at expected move distance)

  Wing Placement: Distance = ATM Straddle Price × 1.0
  Position Sizing: 1-2% of portfolio per trade


  ╔═════════════════════════════════════════════════════════════════════╗
  ║  EXIT RULES (Priority Order)                                       ║
  ╚═════════════════════════════════════════════════════════════════════╝

  1. POST-EARNINGS EXIT (PRIMARY):
     • BMO Earnings → Close SAME DAY at 9:40 AM
     • AMC Earnings → Close NEXT DAY at 9:40 AM
     Bot: "Close if earnings reported BMO today → 9:40 AM"
          "Close if earnings reported AMC yesterday → 9:40 AM"

  2. PROFIT TARGET: 50% of max profit
     • Entry credit $5.00 → Close when worth $2.50

  3. STOP LOSS: 2× credit received
     • Entry credit $5.00 → Close if worth $15.00 (lost $10)

  4. NO FRIDAY AUTO-CLOSE
     • Positions may extend into next week if earnings is Monday


  ╔═════════════════════════════════════════════════════════════════════╗
  ║  BOT CONFIGURATION SUMMARY                                          ║
  ╚═════════════════════════════════════════════════════════════════════╝

  Strategy:          IRON_BUTTERFLY
  Entry Trigger:     earnings_days_away = 3
  Short Delta:       0.50 (ATM)
  Long Delta:        0.16 (1 standard deviation)
  Reward to Risk Ratio : > 20%
  Position Size:     2% of portfolio
  Max Positions:     10
  Closest DTE after Earnings used
  
  Exit Conditions:
  earnings_BMO_today      → 9:40 AM
  earnings_AMC_yesterday  → 9:40 AM
  profit >= 50%           → Close
  loss >= 2× credit       → Close


  ════════════════════════════════════════════════════════════════════
  Run this screener every Tuesday morning. Input all tickers into bot.
  Bot handles everything else. Review results next Tuesday.
  ════════════════════════════════════════════════════════════════════
    """)
    
    return df


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    results = run_screener()
    
    if results is not None and len(results) > 0:
        final_df = print_results_with_rules(results)